In [1]:
import pandas as pd
import import_ipynb
import nbimporter
from utils import prepareTrainingSet3Sec,prepareTrainingSet30Sec,standardization,metrics,Grafico_Before_After,Grafico_Tre_Valori, ConfrontoGrafico_30_e_3, evaluate_Model, prepareTrainingSet30Sec_Arg , prepareTrainingSet3Sec_Arg  #type: ignore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV  
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score

I file CSV per il training e il test set sono stati creati con successo.


In [23]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Prepara i dati
X_train, X_test, y_train, y_test = prepareTrainingSet3Sec()

# Converti le etichette delle classi in valori numerici
label_encoder = LabelEncoder()
y_test = label_encoder.fit_transform(y_test)
y_train = label_encoder.fit_transform(y_train)

# utilizzo la funzione di standardizzazzione presente in utils che usa Standard Scaler
X_train, X_test = standardization(X_train, X_test)

# Creare il modello Logistic
model = XGBClassifier(learning_rate=0.01, n_estimators=600, max_depth=9, min_child_weight=25, subsample=0.8, colsample_bytree=0.8)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
evaluate_Model(y_test, y_pred, y_train, y_pred_train)

---------STATISTICHE TEST----------
Accuratezza: 0.8509
Precision: 0.8497
Recall: 0.8513
F1-score: 0.8495
F2-score: 0.8503
---------STATISTICHE TRAING----------
Accuratezza: 0.9439
Precision: 0.9443
Recall: 0.9439
F1-score: 0.9439
F2-score: 0.9439

=== Analisi ===
Il modello sembra bilanciato e generalizza bene.


In [6]:
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Prepara i dati
X_train, X_test, y_train, y_test = prepareTrainingSet3Sec()

# Codifica le etichette
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

# Standardizzazione
X_train, X_test = standardization(X_train, X_test)

# Definisci la griglia di iperparametri
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

# Configurazione manuale della Grid Search
best_score = 0
best_params = {}
cv = StratifiedKFold(n_splits=5)

# Ciclo su tutte le combinazioni di iperparametri
for max_depth in param_grid['max_depth']:
    for learning_rate in param_grid['learning_rate']:
        for n_estimators in param_grid['n_estimators']:
            for subsample in param_grid['subsample']:
                for colsample_bytree in param_grid['colsample_bytree']:
                    
                    scores = []
                    # Cross-validation manuale
                    for train_idx, val_idx in cv.split(X_train, y_train):
                        X_tr, X_val = X_train[train_idx], X_train[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]
                        
                        model = XGBClassifier(
                            objective='binary:logistic',
                            max_depth=max_depth,
                            learning_rate=learning_rate,
                            n_estimators=n_estimators,
                            subsample=subsample,
                            colsample_bytree=colsample_bytree
                        )
                        model.fit(X_tr, y_tr)
                        scores.append(model.score(X_val, y_val))
                    
                    # Calcola l'accuracy media
                    mean_score = np.mean(scores)
                    if mean_score > best_score:
                        best_score = mean_score
                        best_params = {
                            'max_depth': max_depth,
                            'learning_rate': learning_rate,
                            'n_estimators': n_estimators,
                            'subsample': subsample,
                            'colsample_bytree': colsample_bytree
                        }

print("Best parameters:", best_params)
print("Best cross-validation score:", best_score)

# Addestra il modello finale con i migliori parametri
best_model = XGBClassifier(objective='binary:logistic', **best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

KeyboardInterrupt: 

In [4]:
import xgboost, sklearn
print(xgboost.__version__)
print(sklearn.__version__)

2.1.2
1.6.1
